In [ ]:
import numpy as np

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.synthesis import synth_qft_full
from qiskit_aer import AerSimulator


def make_A(theta):
    qc = QuantumCircuit(1)
    qc.ry(2 * theta, 0)
    return qc.to_gate(label="A")

def make_Q(theta):
    A = make_A(theta)
    qc = QuantumCircuit(1)
    qc.append(A, [0])
    qc.z(0)
    qc.append(A.inverse(), [0])
    qc.z(0)
    return qc.to_gate(label="Q")

def make_qae_circuit(true_a, num_eval_qubits=5):
    theta = np.arcsin(np.sqrt(true_a))
    eval_reg = QuantumRegister(num_eval_qubits, "eval")
    system = QuantumRegister(1, "q")
    classical = ClassicalRegister(num_eval_qubits, "c")
    qc = QuantumCircuit(eval_reg, system, classical)
    A = make_A(theta)
    Q = make_Q(theta)
    # -------------------------------------------
    # 1. Prepare the QAE state A|0>
    # -------------------------------------------
    qc.append(A, [system[0]])
    # -------------------------------------------
    # 2. Put phase-estimation register into
    #    uniform superposition
    # -------------------------------------------
    qc.h(eval_reg)
    # -------------------------------------------
    # 3. Controlled powers
    #
    # Q, Q^2, Q^4, Q^8, ...
    # -------------------------------------------
    for j in range(num_eval_qubits):
        power = 2 ** j
        Q_power = Q.power(power)
        controlled_Q = Q_power.control(1)
        qc.append(
            controlled_Q,
            [eval_reg[j], system[0]]
        )
    # -------------------------------------------
    # 4. Inverse QFT
    # -------------------------------------------
    inverse_qft = synth_qft_full(
        num_eval_qubits,
        inverse=True,
        do_swaps=True
    )
    qc.compose(
        inverse_qft,
        qubits=eval_reg,
        inplace=True
    )
    # -------------------------------------------
    # 5. Measure phase register
    # -------------------------------------------
    qc.measure(eval_reg, classical)
    return qc


def estimate_amplitude(counts, num_eval_qubits):
    M = 2 ** num_eval_qubits
    # QPE produces two symmetric peaks:
    #
    # y and M-y
    #
    # because Q has eigenvalues e^(+i2θ)
    # and e^(-i2θ).
    #
    # Combine these equivalent results.
    combined = {}
    for bitstring, count in counts.items():
        y = int(bitstring, 2)
        canonical_y = min(y, (M - y) % M)
        combined[canonical_y] = (
            combined.get(canonical_y, 0) + count
        )
    # Most likely phase estimate
    y_hat = max(combined, key=combined.get)
    theta_hat = np.pi * y_hat / M
    a_hat = np.sin(theta_hat) ** 2
    return a_hat, y_hat

True amplitude:      0.3
Measured phase bin:  193467
QAE estimate:        0.2999984140427344
Absolute error:      1.5859572655707588e-06


In [7]:
true_a = 0.30
num_eval_qubits = 10
shots = 1

qc = make_qae_circuit(true_a=true_a, num_eval_qubits=num_eval_qubits)
simulator = AerSimulator()
compiled = transpile(qc, simulator)
result = simulator.run(compiled,shots=shots).result()
counts = result.get_counts()
a_hat, y_hat = estimate_amplitude(counts,num_eval_qubits)

print("True amplitude:     ", true_a)
print("Measured phase bin: ", y_hat)
print("QAE estimate:       ", a_hat)
print("Absolute error:     ", abs(a_hat - true_a))

True amplitude:      0.3
Measured phase bin:  189
QAE estimate:        0.3001879000771766
Absolute error:      0.00018790007717661705
